<a href="https://colab.research.google.com/github/huutai-cmyk/Homework-1/blob/main/apptiendien.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


df_raw = pd.read_csv('/content/dl tiendien.csv')

df = df_raw.rename(columns={
    'Số người trong phòng là bao nhiêu?': 'SoNguoi',
    'Số máy lạnh ở phòng là bao nhiêu?': 'SoMayLanh',
    'Phòng có tủ lạnh không?': 'CoTuLanh',
    'Số giờ bật máy lạnh trên ngày?': 'GioMayLanh',
    'Diện tích phòng khoảng bao nhiêu m2?': 'DienTich',
    'Tiền điện trung bình 1 tháng là bao nhiêu vnd?': 'TienDien'
})

df['SoQuat'] = df['Số quạt máy và thời gian sử dụng 1 ngày?'].str.extract('(\d+)').astype(int)

def clean_loainha(x):
    x = str(x).strip().lower()
    if 'trọ' in x: return 'Trọ/Phòng trọ'
    if 'chung cư' in x: return 'Chung cư'
    if 'căn hộ' in x: return 'Căn hộ'
    if 'nhà nguyên căn' in x: return 'Nhà nguyên căn'
    if 'túc xá' in x: return 'Ký túc xá'
    return 'Khác'

df['LoaiNha'] = df['Loại nhà? (trọ,chung cư, căn hộ…)'].apply(clean_loainha)
df = df[['SoNguoi', 'SoMayLanh', 'SoQuat', 'CoTuLanh', 'GioMayLanh', 'DienTich', 'LoaiNha', 'TienDien']]

X = df.drop('TienDien', axis=1)
y = df['TienDien']

numeric_features = ['SoNguoi', 'SoMayLanh', 'SoQuat', 'GioMayLanh', 'DienTich']
categorical_features = ['CoTuLanh', 'LoaiNha']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])
model.fit(X, y)




def predict_bill(songuoi, somaylanh, soquat, cotulanh, giomaylanh, dientich, loainha):
    giomaylanh_thucte = giomaylanh if somaylanh > 0 else 0
    input_data = pd.DataFrame({
        'SoNguoi': [songuoi], 'SoMayLanh': [somaylanh], 'SoQuat': [soquat],
        'CoTuLanh': [cotulanh], 'GioMayLanh': [giomaylanh_thucte], 'DienTich': [dientich], 'LoaiNha': [loainha]
    })
    prediction = model.predict(input_data)[0]
    return f"{prediction:,.0f} VNĐ"


def vao_ung_dung(name):
    if not name or str(name).strip() == "":
        return gr.update(visible=True), gr.update(visible=False), "<span style='color:red; font-weight:bold;'>⚠️ Lỗi: Bạn chưa nhập tên! Vui lòng nhập để tiếp tục.</span>", ""
    else:
        loi_chao = f"### 👋 Xin chào **{name.strip()}**! Hãy nhập thiết bị của bạn vào App nhé:"
        return gr.update(visible=False), gr.update(visible=True), "", loi_chao


with gr.Blocks(theme=gr.themes.Soft(), title="App Dự Đoán Tiền Điện") as app:
    with gr.Column(visible=True) as man_hinh_chao:
        gr.HTML("""
            <div style="display: flex; justify-content: center; align-items: center;
                        height: 200px; background-color: #4F46E5; color: white;
                        font-size: 50px; font-weight: bold; border-radius: 20px;
                        box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); margin-top: 20px;">
                HELLO
            </div>
        """)

        gr.Markdown("<br>")
        o_nhap_ten = gr.Textbox(label="Vui lòng cho App biết tên của bạn:", placeholder="Nhập tên của bạn vào đây...")
        thong_bao_loi = gr.Markdown(value="")
        nut_enter = gr.Button("🚀 Enter", variant="primary", size="lg")
    with gr.Column(visible=False) as man_hinh_chinh:
        loi_chao_ca_nhan = gr.Markdown("### 👋 Xin chào!")
        with gr.Row():
            with gr.Column():
                w_songuoi = gr.Slider(minimum=1, maximum=10, step=1, value=2, label="👥 Số người ở")
                w_somaylanh = gr.Slider(minimum=0, maximum=5, step=1, value=1, label="❄️ Số máy lạnh")
                w_soquat = gr.Slider(minimum=0, maximum=10, step=1, value=2, label="🎐 Số quạt")
                w_cotulanh = gr.Radio(choices=["Có", "Không"], value="Có", label="🧊 Có tủ lạnh không?")
            with gr.Column():
                w_giomaylanh = gr.Slider(minimum=0, maximum=24, step=1, value=4, label="⏱️ Số giờ bật ML/ngày")
                w_dientich = gr.Number(value=25, label="📐 Diện tích phòng (m2)")
                w_loainha = gr.Dropdown(
                    choices=['Trọ/Phòng trọ', 'Chung cư', 'Căn hộ', 'Nhà nguyên căn', 'Ký túc xá'],
                    value='Trọ/Phòng trọ',
                    label="🏠 Loại nhà"
                )

        nut_du_doan = gr.Button("🔮 Dự đoán tiền điện", variant="primary", size="lg")
        ket_qua_du_doan = gr.Textbox(label="💡 ƯỚC TÍNH TIỀN ĐIỆN MỖI THÁNG", lines=2)


    nut_enter.click(
        fn=vao_ung_dung,
        inputs=[o_nhap_ten],
        outputs=[man_hinh_chao, man_hinh_chinh, thong_bao_loi, loi_chao_ca_nhan]
    )


    o_nhap_ten.submit(
        fn=vao_ung_dung,
        inputs=[o_nhap_ten],
        outputs=[man_hinh_chao, man_hinh_chinh, thong_bao_loi, loi_chao_ca_nhan]
    )


    nut_du_doan.click(
        fn=predict_bill,
        inputs=[w_songuoi, w_somaylanh, w_soquat, w_cotulanh, w_giomaylanh, w_dientich, w_loainha],
        outputs=[ket_qua_du_doan]
    )


app.launch(share=True)

<>:25: SyntaxWarning: invalid escape sequence '\d'
<>:25: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_515/2474804979.py:25: SyntaxWarning: invalid escape sequence '\d'
  df['SoQuat'] = df['Số quạt máy và thời gian sử dụng 1 ngày?'].str.extract('(\d+)').astype(int)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a1eb392dc5529ccdf8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Mục mới

# Mục mới